# KosenMap Website 取扱説明書 —— 目次

**この Website(公開ページ・管理画面・本番ホスト)を動かすための取扱説明書。**
読むだけでなく、**書いてあるコマンドをそのセルから実行できる。**

対象は本番 `ito4.jp`(`/opt/kosenmap`)と、この PC の `server/scripts`。
**繋ぐ利用者は 2 つ**(2026-09-18 に分けた。[12](12-hardening-2026-09-15.ipynb) §7-4 B): 配備・`%%host`・控えは **`kmops`**(鍵 `~\.ssh\km_ops`。docker あり・**sudo なし**)、`sudo` が要るセルは **`km`**(鍵 `~\.ssh\km_vps`。**docker なし**)。
Android アプリは別リポジトリなので、ここには入っていない。

| ノートブック | 何が書いてあるか |
|---|---|
| [00-start](00-start.ipynb) | **この取扱説明書の使い方。** 準備と、セルの型 |
| [01-daily-check](01-daily-check.ipynb) | **日々の確認。** 自己検査・ホストの様子・メールが届いているか |
| [02-deploy](02-deploy.ipynb) | **配備。** 下見・配備・後片付け・困ったとき |
| [03-backup](03-backup.ipynb) | **バックアップと復元。** 取る・開く・週次タスク・添付の復号・戻す |
| [04-host-jobs-and-mail](04-host-jobs-and-mail.ipynb) | **ホストの定期処理とメール。** cron の時刻表・届くメールの一覧・試しに送る |
| [05-containers](05-containers.ipynb) | **コンテナの更新と追加。** Logto の上げ方・固定・足し方 |
| [06-emergency](06-emergency.ipynb) | **もしものとき。** まず叩く1本と、症状別の見どころ |
| [07-map-qr](07-map-qr.ipynb) | **地図の配信と QR。** |
| [08-architecture](08-architecture.ipynb) | **どう出来ているか。** 構造・設定の読み方・落とし穴・設計の約束 |
| [09-new-host](09-new-host.ipynb) | **新しいホストを作る・移す。** VPS の構築から切り替えまで |
| [10-security-review-2026-09-14](10-security-review-2026-09-14.ipynb) | **診断の報告書(2026-09-14)。** 何を見つけて何を直したか・本番で実行する手順 |
| [11-getting-started](11-getting-started.ipynb) | **新しく使う人の入口。** 全体の図・部品の役割・管理画面・Android アプリ・用語集 |
| [12-hardening-2026-09-15](12-hardening-2026-09-15.ipynb) | **多層防御の底上げ(2026-09-15)。** 網の分割・read_only・Soketi の Node 24・Logto の DB 利用者・管理画面の別オリジン・本番への当て方 |
| [13-local-env](13-local-env.ipynb) | **ローカル環境(LAN の検証機)。** `.env` の `KM_ENV=local` で、Let's Encrypt を使わずに本番と同じ構成を立てる |
| [14-domain-ito4](14-domain-ito4.ipynb) | **ito4.jp に統一する(2026-09-17)。** 旧 ito8795.com は同時に手放す・audience も https://ito4.jp/api へ・管理画面は admin.ito4.jp・外部スキャンの指摘 |
| [15-staff-org](15-staff-org.ipynb) | **教職員の自動付与(2026-09-18〜)。** 学校ドメインの JIT で Logto の組織へ入れ、組織ロールで「地図の錠を通る・教職員氏名を見る・閲覧不可の地点を見る」を与える |
| [17-cloudflare](17-cloudflare.ipynb) | Cloudflare に載せるときの計画(書くだけ・当てていない) |
| [18-room-shape-research](18-room-shape-research.ipynb) | 写真から部屋の形を作る(調査の報告) |
| [19-security-fixes-2026-10-06](19-security-fixes-2026-10-06.ipynb) | 2026-10-05 の診断の反映(直したもの・確かめたこと・残り) |
| [20-indoor-positioning](20-indoor-positioning.ipynb) | 現在地の測位の再構築(候補 → 相互検証 → 粒子フィルタ・パラメータ・比較実験・RTT・ARCore) |

記録として残している文書:

| 文書 | 中身 |
|---|---|
| [status.md](status.md) | **いまどうなっているか。** 実測値・踏んだ罠・残作業 |
| [plan.md](plan.md) | **これから何をするか。** 優先順と、やらないと決めたこと |
| [../Old/docs/](../Old/docs/) | 以前の手順書(md)。**細部の経緯はここ**。ノートブックに移した内容の元 |
| [17-cloudflare](17-cloudflare.ipynb) | Cloudflare に載せるときの計画(書くだけ・当てていない) |
| [18-room-shape-research](18-room-shape-research.ipynb) | 写真から部屋の形を作る(調査の報告) |

In [ ]:
# 最初に1回だけ実行する(%%ps / %%host / %%terminal が使えるようになる)
import sys, pathlib
for _d in (pathlib.Path.cwd(), pathlib.Path.cwd() / 'docs', pathlib.Path.cwd() / 'server' / 'docs'):
    if (_d / 'km_nb.py').exists():
        sys.path.insert(0, str(_d))
        break
import km_nb
km_nb.load()


# 20. 現在地の測位の再構築(2026-10-06)

利用者の指示「現在の Wi-Fi の位置特定が微妙なので再構築」「全てを使用して平均を出すではなく、それぞれの長所を利用して
**候補位置を生成 → 信頼度を評価 → 融合**」。出典は利用者から受け取った指示書(`Android Wi-Fi・ARCore統合型屋内測位システム 再構築指示書`。リポジトリには入れていない)。

## 1. なぜ作り直したか

| 前の作り | 問題 |
|---|---|
| ハイブリッド法 = 指紋と三辺測量を**固定の割合(0.7 / 0.8)で混ぜる** | どちらかが大きく外れても、そのまま混ざる |
| 信頼度 = いちばん近い学習地点との dB 差だけ | 方式どうしの食い違い(外れ値)を見ていない |
| 三辺測量はルーターごとの係数が無いと出ない。最小二乗は線形化 1 回・重みなし | ほとんどの階で出ない・遠いルーターの誤差に引っ張られる |
| 時間方向は 1 次元のカルマン + PDR。壁は「前の点から線を越えたか」だけ | 地図(壁・歩ける所)を確率として使っていない |
| RTT・ARCore・機械学習・粒子フィルタは無い | — |

評価の記録(2026-10-06 時点): 3F の 1 地点で補正後 9.8 m。学習地点は 3F の 7 地点だけ。

## 2. 全体の形

```
Wi-Fi スキャン ─ RssiSmoother(BSSID ごとの指数移動平均・中央値・ばらつき)
  ├─ 三辺測量 → 重み付き最小二乗(Gauss-Newton。壁の数で減衰を補正)
  ├─ AP 加重重心
  ├─ 指紋 + 加重 KNN(近傍ごとに分布の山を出す)
  ├─ Wi-Fi RTT(端末と AP が対応しているときだけ)
  └─ 機械学習回帰(比較用)
IMU ─ PDR(歩数と方角)          ┐ 予測(移動量と不確かさ)
ARCore(AR 測位モード)─ Motion ┘ Depth: 前方の壁までの距離
        ↓ どれも同じ形(PositionEstimate: x, y, 階, σ, 信頼度, 時刻, 方式, 理由)
相互検証  合意点(重み付き幾何中央値)から外れた候補を外れ値に・一致すれば信頼度を上げる
3D 空間  地図の壁・歩ける線・ルーターの位置。歩ける所から遠い候補を落とす
確率分布  採用した候補をガウスの混合に(「位置A 55% / 位置B 30%」)
粒子フィルタ  移動で予測 → 確率分布で重み → 壁を越えた粒子は重みを下げる → 再標本化(本番)
EKF  同じ入力で並走(比較用)
```

**「ハイブリッド法」= この構成全体**(指示書 §20・§22)。設定の値はそのまま(移行なし)。来場者のアプリは常にこれ。
管理アプリでは比較のため単独の方式も選べる(三辺測量 / 三辺測量+最小二乗 / AP加重重心 / セントロイド / 指紋 KNN / 指紋 加重KNN / RTT / 機械学習)。融合の方法は「粒子フィルタ(既定)/ EKF(比較)」。

## 3. 方式ごとの役割(アプリ `app/src/main/java/com/ito/kosenmap/positioning/`)

| 方式 | 役割 | 信頼度の決め方 | 融合に入れるか | ファイル |
|---|---|---|---|---|
| 三辺測量 | RSSI → 距離 → 位置 | 台数・残差・校正済みの割合 | 比較用(A) | `WifiEstimators.kt` |
| 三辺測量 + 最小二乗 | 距離の誤差の二乗和を最小に(重み = 距離の誤差の見込みの逆二乗) | 同上(重み付きは高め) | **本番** | 同上 |
| AP 加重重心 | ルーターの位置の重み付き重心 | 台数(単独では精度が低いので低め) | **本番** | 同上 |
| 指紋 + 加重 KNN | 学習地点との電波の差で近い K 件を重み付き平均。近傍ごとに山を出す | 最も近い差(dB)・学習回数 | **本番** | 同上 |
| 指紋 + KNN | 普通の KNN | 同上より少し低い | 比較用(D) | 同上 |
| Wi-Fi RTT | 往復時間で距離 → 重み付き最小二乗 | 台数・残差(RSSI よりずっと高い) | **本番**(使えるとき) | `RttEstimator.kt`・`WifiRttController.kt` |
| 機械学習回帰 | RSSI のベクトル → (x, y)。浅い回帰木の森を端末で学習 | 学習に使わなかった地点での誤差 | 比較用(F) | `MlRegression.kt` |
| PDR | 歩数と方角で移動を予測 | 歩幅と方角の誤差の割合 | 予測 | `PedestrianDeadReckoning.kt` |
| ARCore | カメラと IMU で移動を測る・前方の壁までの距離 | 移動量の誤差の割合(小さい) | 予測・Depth | `ArPositioningSession.kt`・`ArGeometry.kt` |
| 3D 空間 | 歩ける所から遠い候補を落とす・壁の数・粒子の壁越え・Depth と比べる | — | 制約 | `SpaceModel.kt` |
| 相互検証 | 合意点からの距離 / √(σ² + σ合意²) が閾値を超え、味方が居ない候補を外れ値に | 一致する数に応じて上げる | — | `CrossValidator.kt` |
| 確率的位置推定 | 採用した候補をガウス混合に | 相互検証のあとの信頼度が重み | — | `CandidateMixture.kt` |
| 粒子フィルタ | 時系列の融合(本番) | — | — | `ParticleFilter.kt` |
| EKF | 比較用 | — | — | 同上 |

**外れ値の例**(指示書 §2 原則 3): 三辺測量 (25, 2)・指紋 (10.4, 6.0)・重心 (10.3, 5.9)・PDR (10.5, 6.1)(m)なら、
三辺測量だけが外れ値になり、残りは一致しているので信頼度が上がる(JVM の試験 `theFarTrilaterationIsAnOutlierWhenTheOthersAgree`)。
候補が 2 つだけで食い違うときは、どちらが正しいか言えないので、**両方の信頼度を下げるだけ**で外れ値にはしない。

## 4. パラメータ(29 項目)

**正本はアプリの `positioning/PositioningParams.kt` の表**。サーバーの `lib/positioning-params.php` も同じ表を持ち、
check.php の `positioning-params` とアプリの試験が、キー・範囲・既定を突き合わせる(片方だけ足すと、配っても効かない・配れない)。

| 部分 | 項目 | キー | 範囲 | 既定 | 何に効くか |
|---|---|---|---|---|---|
| 電波の平滑化 | 指数移動平均の新しい値の重み | `rssi_ema_alpha` | 0.05〜1.0 | 0.5 | 大きいほど新しいスキャンを信じる(反応が速いがぶれる)。 |
| 電波の平滑化 | 中央値とばらつきを見る時間 | `rssi_window_seconds` | 2.0〜120.0 秒 | 20.0 | この時間内のスキャンで、BSSID ごとの中央値とばらつき(分散)を出す。ばらつきは三辺測量の重みに使う。 |
| 距離のモデル | 1 m での電波の強さ(既定) | `path_loss_tx_dbm` | -70.0〜-10.0 dBm | -40.0 | ルーターに個別の値が無いときに使う。実測で校正すると三辺測量が良くなる。 |
| 距離のモデル | 減衰の指数 n(既定) | `path_loss_exponent` | 1.5〜5.0 | 2.8 | 屋内は 2.5〜3.5 が多い。大きいほど同じ dB の差を短い距離と見る。 |
| 距離のモデル | 壁 1 枚の減衰 | `wall_attenuation_db` | 0.0〜15.0 dB | 4.0 | ルーターとの間にある壁の数ぶん、電波が弱まったとみなして距離を短く直す(地図の壁の線で数える)。 |
| 三辺測量 | 使うルーターの最少台数 | `trilateration_min_anchors` | 3.0〜10.0 台 | 3.0 | これより少ないと三辺測量は出さない。 |
| 三辺測量 | 最小二乗の反復回数 | `ls_iterations` | 1.0〜50.0 回 | 12.0 | Gauss-Newton の反復。重み付き最小二乗で距離の誤差の二乗和を小さくする。 |
| フィンガープリント | 近傍の数 K | `knn_k` | 1.0〜10.0 件 | 4.0 | 加重 KNN で混ぜる学習地点の数。学習地点が少ないうちは小さく。 |
| フィンガープリント | 見えない BSSID の扱い | `fingerprint_missing_dbm` | -110.0〜-80.0 dBm | -95.0 | 片方に無い BSSID は、この強さで見えていたとみなして差を取る。 |
| フィンガープリント | 比べる BSSID の下限 | `fingerprint_compare_min_dbm` | -95.0〜-60.0 dBm | -85.0 | どちらかでこれより強い BSSID だけを比べる(弱い BSSID は出たり消えたりする)。 |
| フィンガープリント | 信頼度「高」の電波の差 | `confidence_high_db` | 2.0〜20.0 dB | 8.0 | いちばん近い学習地点との差がこれ以下なら、指紋の信頼度を高くする。 |
| フィンガープリント | 学習範囲の外とみなす電波の差 | `coverage_db` | 5.0〜30.0 dB | 14.0 | これを超えたら学習した範囲の外にいる見込み(指紋の信頼度を大きく下げる)。 |
| セントロイド | 加重重心の強さの指数 | `centroid_exponent` | 0.1〜3.0 | 1.0 | 重み = 10^((RSSI−最強)/10 × 指数)。大きいほど強いルーターに寄る。 |
| Wi-Fi RTT | Wi-Fi RTT を使う | `rtt_enabled` | 0.0〜1.0(0/1) | 1.0 | 端末と AP が対応しているときだけ測る。使えなければ RSSI だけで測る。 |
| Wi-Fi RTT | RTT で位置を出す最少の AP 数 | `rtt_min_responders` | 2.0〜10.0 台 | 3.0 | これより少ないと RTT の位置は出さない(距離は相互検証にだけ使う)。 |
| PDR・ARCore | 歩幅の誤差の割合 | `pdr_step_noise_ratio` | 0.0〜1.0 | 0.3 | 1 歩ごとに歩幅のこの割合ぶん位置の不確かさが増える。 |
| PDR・ARCore | 方角の誤差 | `pdr_heading_noise_deg` | 0.0〜45.0 度 | 12.0 | 粒子を進める向きを、この標準偏差でばらつかせる(地磁気の乱れを吸収する)。 |
| PDR・ARCore | ARCore の移動量の誤差の割合 | `ar_translation_noise_ratio` | 0.0〜0.5 | 0.05 | AR 測位モードの移動量は PDR よりずっと正確なので小さい。 |
| PDR・ARCore | Depth(前方の壁)の効き | `ar_depth_weight` | 0.0〜1.0 | 0.5 | 前方の壁までの距離が地図と合わない粒子の重みを下げる強さ。0 で使わない。 |
| 相互検証 | 外れ値とみなす σ の倍数 | `outlier_gate_sigma` | 1.0〜10.0 倍 | 3.0 | 合意点からの距離が、不確かさのこの倍より大きければ外れ値。 |
| 相互検証 | 外れ値とみなす最小の距離 | `outlier_min_m` | 1.0〜30.0 m | 6.0 | これより近ければ外れ値にしない(σ が小さすぎるときの誤判定を防ぐ)。 |
| 相互検証 | 歩ける所からの最大距離 | `walkable_max_m` | 0.5〜20.0 m | 4.0 | 道・階段の線からこれより離れた候補は、物理的にありそうにないとして外す(線が無い階では使わない)。 |
| 相互検証 | 部屋の広さ(半径) | `room_radius_m` | 1.0〜20.0 m | 5.0 | 部屋・出入口の点のまわりこの半径は歩ける所として扱う(部屋の中の候補を外さず、通路へ寄せない)。 |
| 粒子フィルタ | 粒子の数 | `particle_count` | 100.0〜3000.0 個 | 500.0 | 多いほど滑らかだが計算が増える。 |
| 粒子フィルタ | 歩数が取れないときの移動の見込み | `particle_drift_mps` | 0.0〜3.0 m/秒 | 0.5 | 歩数が取れなくても人は動きうるので、時間に応じて粒子を広げる。 |
| 粒子フィルタ | 再標本化する有効粒子数の割合 | `particle_resample_ess` | 0.1〜1.0 | 0.5 | 重みの偏りでこの割合を下回ったら、粒子を引き直す。 |
| 粒子フィルタ | 歩ける所から離れた粒子の重み | `particle_off_walk_penalty` | 0.0〜1.0 倍 | 0.3 | 歩ける所からの最大距離を超えた粒子の重みに掛ける。 |
| 粒子フィルタ | 壁を越えた粒子の重み | `particle_wall_penalty` | 0.0〜1.0 倍 | 0.02 | 1 回の移動で壁の線を越えた粒子の重みに掛ける(地図の壁が間違っていても固まらないよう 0 にはしない)。 |
| EKF(比較) | EKF の移動の雑音 | `ekf_process_noise_mps` | 0.0〜3.0 m/秒 | 0.5 | 粒子フィルタとの比較用。時間あたりの不確かさの増え方。 |

### 編集と配り方(管理アプリ)

1. 設定 →「測位のパラメータ(管理)」→「測位のパラメータを見る・変える・配る」
2. 「値を見る・変える」→ 部分 → 項目。一覧には今の値と、既定・配っている値と違えばそれも出る
3. 値を入れて「次へ(確認)」→ **変更前 → 後・範囲の外なら丸めたこと・効くところ**を見て「変える」。この端末だけに効く(次のスキャンから)
4. 地図の左上(「各種計算方法の生表示」)で、方式ごとの点・外れ値・確率・粒子・既定と違うパラメータを見ながら確かめる
5. 良ければ「全員の端末に配る」(変わる項目の一覧を見て確認)。一般のアプリは**次に地図を取得・更新したとき**から使う
6. 戻すとき: 「配るのをやめる」(全員が既定へ)/「この端末の値を既定に戻す」/「配っている値を読み込む」

サーバー: `api/positioning-params.php`(GET は誰でも・POST は管理者のトークン)。値は専用の表 `km_positioning_params`。
操作ログに「配った・やめた」を残す(詳細は既定と違う項目の数だけ)。nginx の km_api で回数を絞る。

## 5. 表示の読み方(管理アプリの「各種計算方法の生表示」)

- 地図: 灰色の小さな点 = 粒子。方式ごとの点と σ の円(色は方式)。**✕ = 外れ値**。淡い点 = 比較用(融合に入れない方式)
- 左上の欄:
  - 「融合: 粒子フィルタ・採用 3/4・一致度 67%・計算 4.2ms」
  - 「● 指紋(加重KNN) 信頼 72%(元 60%)σ 3.1m」= 相互検証のあとの信頼度と、その方式が自分で見積もった信頼度
  - 「✕ 三辺測量+最小二乗: 他の方式と食い違う(外れ値)(合意点から 14.2m)」
  - 「確率: 位置A 55% / 位置B 30%」・「粒子 300 個表示・有効 62%」・「EKF(比較): 粒子との差 1.2m」
  - 「Wi-Fi RTT: この端末は非対応」など、RTT が使えない理由
  - 「学習地点 7 件(少ない…)」—— 学習地点が 15 未満の階で出る
  - 「パラメータ(既定と違う): …」

## 6. 比較実験(指示書 §19・§24)

管理マップで地点を選んで精度の評価を登録すると、選んでいる方法の結果に加えて、**同じ時点の構成 A〜M と EKF を全部**記録する。

| 構成 | 中身 |
|---|---|
| A / B | 三辺測量 / 三辺測量 + 最小二乗 |
| C | AP 加重重心 |
| D / E | 指紋 + KNN / 指紋 + 加重 KNN |
| F | 機械学習回帰(学習地点 8 未満の階では出ない) |
| G | Wi-Fi RTT(使えないときは「出せなかった」) |
| H / I | PDR だけ / ARCore だけ(本番の推定から 1 度だけ出発し、その後は移動だけ。I は AR 測位モードを使ったときだけ) |
| J / K / L | Wi-Fi(RSSI)+ PDR / + ARCore / + PDR + ARCore(粒子フィルタ・3D 空間の制約なし) |
| M | 全方式 + 粒子フィルタ(本番。3D 空間の制約あり) |
| EKF | 全方式 + EKF(比較) |

- 評価の画面: 構成ごとに件数・平均・RMSE・95%・最大・計算時間(1 回の平均)・ぶれ(Wi-Fi が届くたびの動きの平均)・出せなかった回数。記録時の使用メモリの平均
- CSV(精度評価 CSV のエクスポート): 1 件につき、選んでいた方法の行と、構成ごとの行(algorithm が「比較 M …」)。
  末尾に `variant, available, computeMillis, jitterMeters, memoryMb` を足した。出せなかった構成は位置と誤差の列が空
- 「全部を組み合わせたら本当に良くなったか」は、**同じ地点で M と A〜L を比べる**。学習地点を増やす前と後でも比べる

## 7. Wi-Fi RTT と ARCore の条件

| | 条件 | 使えないとき |
|---|---|---|
| Wi-Fi RTT | 端末が RTT 対応(Pixel など)・AP が 802.11mc に応じる・位置情報と付近の Wi-Fi の許可 | 理由を表示して RSSI だけで測る。学校の AP が 802.11mc に応じなければ使えない |
| AR 測位モード | ARCore 対応の端末・「Google Play 開発者サービス(AR)」・カメラの許可。地図の右上の「AR」で入り切り | ボタンを出さない(非対応)・導入を尋ねる。LDPlayer では使えない |

- AR 測位モードは**端末を前に向けて持つ**。追跡が外れたら PDR に戻る(両方は足さない)。地図の画面を離れる・アプリが裏に回ると止まる(電池と発熱)
- 方位: ARCore の向き(始めたときに勝手に決まる)と、端末の回転ベクトルから出した**カメラの向きの方角**を比べて合わせる
- Depth は対応端末だけ。端末をほぼ水平に前へ向けているとき、0.5 秒に 1 回、画面中央の先の物までの距離を地図の壁と比べる(保存しない)

## 8. 確かめたこと・まだのこと

| 確かめたこと | どうやって |
|---|---|
| 各方式・相互検証・確率・粒子フィルタ・EKF・RTT・機械学習・AR の幾何・比較の構成・集計・CSV・パラメータの読み順と配り方 | JVM の試験(`positioning/` の試験 5 本と既存の試験)。両フレーバーで全件通過 |
| サーバーの検め方・口の守り・アプリとの表の一致 | check.php `positioning-params` |
| 両フレーバーの APK の組み立て(ARCore 込み) | gradlew assemble |

**まだ(実機が要る)**: 地図の上の表示・パラメータの確認ダイアログ(LDPlayer で可)、RTT・AR 測位モード・歩いての比較(Pixel 7a。利用者の許可が要る)。
学習地点が 7 つしかないので、**どの方式でも学習範囲の外は弱い**。学習地点を増やしてから A〜M を比べるのが本筋。

## 9. 本番に当てるには(利用者)

- サーバー: `deploy-to-host.ps1 -Action deploy -Yes`(lib・api・admin-log・check)→ nginx の設定を変えたので `-Action restart -Services reverse-proxy`(または `-Action up`)
- アプリ: リリース版を作って署名し、管理画面「ダウンロード」で差し替える(自動アップデートで届く)
- AR 実測・画像(12・13)も同じ配備で入る(pi/ar-capture.php・管理画面「AR の撮影」・nginx の 2 つの口・host-backup.sh)

## 10. 学習データはサーバーに置く(2026-10-06 夜、利用者の指示)

「学習データは基本的にサーバーに保存し、データをダウンロードしてクライアントだけで処理する」。

| | どうしたか |
|---|---|
| それまで | 学習データは**管理端末の中だけ**。サーバーの表(`km_map_fingerprints`)は 0 行で、**一般のアプリには学習データが届いていなかった**(指紋の方式が働いていなかった) |
| 置き場 | Website の `api/learning.php` と表 `km_learning_samples`・`km_learning_evaluations`(`lib/learning-data.php`)。**サーバーは保存するだけで計算しない** |
| 送る単位 | **学習 1 回ごとの生の記録**(利用者の決定)。管理端末が何台あっても上書きで消えない。同じ記録は 2 度入らない |
| 合わせ方 | アプリが全部の記録を落とし、地点ごとに**測った順に重ねて**学習地点の値を作る(端末で学習したときと同じ重ね方。`LearningSync.kt` の `fingerprintsFromSamples`)。機械学習も端末で学ぶ |
| 落とし方 | 地図の取得・更新のとき(と起動したとき)に、**前回から変わった分だけ**(通し番号)。消された記録も知らされる。一般のアプリは地図のアクセスコードで読む(コードは URL に載せず本文で送る) |
| 送り方 | 管理アプリは学習登録のたびに「未送信」に積み、送れたら外す(圏外でも失わない)。最初の同期で、端末にあった学習地点(7 地点)を 1 度だけ送る |
| 評価 | 精度の評価(A〜M の比較)も置く。**管理アプリだけ**が読み書きする |
| リセット | 「Wi-Fiの学習状況をリセット」で学習データ・評価を消すと、**サーバーの分も消す(全員の端末から)**。端末の控えから「元に戻す」と、サーバーにも置き直す |
| 状態 | 設定の「Wi-Fiの学習状況をリセット」の説明に「サーバーと同期: 最後・記録の数・未送信・失敗の理由」 |

## 11. アプリの版は 1 → 2 → 3

それまでの版番号はビルドした日時から作っていた(400837 など)。いまは `app/version.properties` の番号(版)で、
**リリース版の組み立てが成功したら 1 つ上がる**(次のリリースの番号になる)。画面の表示は「2 (261007.1453)」のように、版とビルドした日時。

**Android が比べる内部の番号(versionCode)は 1000000 + 版**(2026-10-07、利用者の決定)。本番で配っていた APK は日時から作った
400837(10/5)だったので、素直に 2 にすると番号が下がり、入っている端末に上書きできず、自動更新も「古い」と見て知らせなかった。
配布(管理画面「ダウンロード」)では、**版番号の欄は空けておく**(APK の中から 1000002 を読む)。表示名の欄に「2」と入れる。

| 版 | 内部の番号 | 作った日 |
|---|---|---|
| 2 | 1000002 | 2026-10-07(一般用 `KosenMap-2.apk`・管理用 `KosenMap-admin-2.apk`。Android の `release/2/`) |

## 12. AR 実測 —— ARCore で地図の誤差を直す(2026-10-06 夜、利用者の指示)

手描きの地図には「縦横・向きで縮尺が違う」「北がずれている」「地点・壁の位置がずれている」誤差がある。
ARCore は歩いた距離と向きを数 % の誤差で測れるので、**測る道具**にして直す候補を出す。**直すかどうかは管理者が決める**(自動では変えない)。

**使い方(管理アプリ)**: 地図で AR 測位モードを ON →「実測」→ 始める → 地点の真上に立って地図でその地点を押し「ここに印を付ける」→
端末を前に向けたまま次の地点へ歩いて印 →(向きを変えて何本か。できれば最初の地点に戻って印)→「終える」。

| 直すもの | どう出すか(アプリ `positioning/ArSurvey.kt`) | 反映先 |
|---|---|---|
| 縦横・向きの縮尺 | 続いた 2 つの印の**直線距離**(AR の座標どうしなので方位の誤差に左右されない)を実測区間に。2 m 未満・途中で追跡切れ・回り道でずれが 5 % を超えそうなものは使わない。同じ 2 点を何度か歩けば平均 | 「北と距離の補正」の実測区間(`withCalibrationSegment`) |
| 北 | 区間ごとに「AR の向き + 方位とのずれ」と「地図の上の向き」の差を出し、長さで重みを付けた円周の平均。ばらつきが 20° を超えるか区間 1 本なら「確かでない」 | 階ごとの「地図の上の方角」 |
| 地点の位置 | 印の AR の位置を地図へ重ねる(5 地点以上はアフィン = 地図全体の歪みは変換が吸う、それ未満は相似)。**4 地点以上のとき**、1 つ抜きで当て直しても 1.5 m 以上外れる地点を「動かす」候補に | 管理の地図(`moveNodeTo`。線の端も追う)して保存 |
| 壁 | 深度で当たった点を地図に移し、1 m の目に 3 回以上たまった所だけ「壁が無いのに当たった」「壁の向こうまで見えた」 | 表示だけ(線は管理者が直す) |

- 結果の画面で項目ごとに選び、「反映する」で**変更前 → 後**を確かめてから入れる。直後なら「元に戻す」で戻せる
- 補正はその端末だけに入る。全員に配るのは今までどおり設定の「北と距離の補正」→「全員の端末に配る」
- 「同じ地点に戻ったときのずれ」が、その歩き方の ARCore のずれの目安(ふつう 1 % 前後)
- 方位(北)は屋内の磁気の乱れを受ける。ばらつきと区間ごとの外れを並べて出すので、それを見て決める
- 記録(軌跡・印・深度の点)は端末(`ar_surveys/`)とサーバー(下の 13)に置く。「前の記録」から開き直せる(別の端末で撮ったものも)。**計算は端末**

## 13. ARCore の画像を保存する(2026-10-06 夜、利用者の決定)

「AR 実測」を始めるときに「画像も保存する」を選ぶと、3D・疑似ストリートビュー用に画像を撮る。

| | どうしたか |
|---|---|
| 撮る間隔 | 前の 1 枚から 0.3 m 動くか 10° 回り、0.5 秒たったとき。速く回している(60°/秒 超)ときは撮らない(ぶれる) |
| 画像 | カメラの読み出しの向きのまま JPEG(画質 90。3MB を超えたら 70)。管理アプリは CPU 画像を 1920×1080 までの最大にする |
| 付帯情報 | 1 枚ごとにカメラの位置・向き(`Camera.getPose`)と内部の値(fx・fy・cx・cy)。深度に対応する端末は 16bit の深度(mm)の PNG も |
| 上限 | 1 回 600 枚・1 枚 3MB・深度 1MB(アプリとサーバーで同じ。check.php の ar-capture が突き合わせる)。端末の未送信は 2GB まで |
| 置き場 | **サーバー(管理者だけ)**。Website の `api/ar-capture.php`・`lib/ar-capture.php`、画像は `uploads/ar-captures/<ハッシュ>`(外からは読めない)。**地図の配信と一般のアプリには載せない** |
| 送り方 | 終えたらすぐ、残りは地図の更新のときに。記録 → 画像の順に 1 枚ずつ。**サーバーが同じハッシュを返したら端末の分を消す**。途中で失敗したら続きは次の機会 |
| 人が写ったら | **PC でぼかす**(利用者の決定)。サーバーには元の画像のまま置くので、処理が済んだら管理画面で消す |
| 控え | 毎日の控え(`host-backup.sh`)には**画像を入れない**(最大 5GB・顔が写った元の画像を何世代も残さないため)。表の行は入るが、zip は無いファイルを飛ばす |
| 全体の上限 | 5GB(本番の空きは 72GB。2026-10-06 に実測) |

**PC で 3D を作る**

1. 管理画面「AR の撮影」→「zip を落とす」(撮影ごと。圧縮しない zip)
2. `scripts\ar-blur-faces.ps1 -Zip <zip>` で展開して顔をぼかす。初めは Python と `py -m pip install deface` を入れる。
   ぼかした画像で元の画像を上書きし、失敗したら展開したフォルダごと消す(ぼかしていない画像を残さない)。`-RemoveZip` で zip も消す
3. ぼかしの見落とし(横顔・小さな顔)が無いか一通り見る
4. RealityScan で `images` を読み込み → 位置合わせ → メッシュ → 書き出し。COLMAP なら `sparse/0` に既知の姿勢(世界は AR の座標・y が上・m)
5. Blender で整える
6. 管理画面で撮影を消す

zip の中身: `images/`・`depth/`・`frames.json`(1 枚ごとの姿勢と内部の値)・`sparse/0/*.txt`(COLMAP)・`survey.json`(AR 実測の記録)・`README.txt`。

## 14. 本物の MariaDB で確かめた(2026-10-06 夜)

この PC には MariaDB が無いので、**本番ホストの上に、本番と切り離した使い捨ての MariaDB**(本番と同じ `mariadb:11.4`・別の網・256MB)を立て、
`kosenmap-web` の像の PHP から送った `lib/` を読んで試した。終わったら DB・網・ファイルを消し、残りが 0 件であることを確かめた。本番の DB・.env・データには触れていない(sudo なし)。

| 試したもの | 件数 | 主な中身 |
|---|---|---|
| 学習データ・評価(`lib/learning-data.php`) | 37 | 置く・送り直しても増えない・消して送り直すと生き返る(新しい番号)・400 行と 600KB ごとに区切る・消した印・評価・番号が重ならない |
| アクセスコードの照合 | 6 | 正しいコード(区切り・小文字も)・空 400・違う 401・止めた 410・期限切れ 410・外れが続くと 429 |
| 測位のパラメータ(`lib/positioning-params.php`) | 13 | 足りない項目を既定で埋める・整数を丸める・範囲外/知らない項目/数でない値を断る・置き換え・やめる・壊れた値は配らない |
| AR の記録と画像(`lib/ar-capture.php`) | 31 | 記録より先に画像は置けない・送り直し・置き換えで前のファイルを消す・上限・一覧と大きさ・zip の中身(取り出すと元と同じ)・ほかの撮影が使う画像は消さない |

1 ページの量は 600KB を少し超える(619KB。1 行足してから量を見るため)。アプリは 1MB まで読むので問題ない。

## 15. 部屋の中でも測位する(2026-10-08)

3F の部屋「1-3」にいても、現在地が通路の先の階段のそばに出た。測位は「歩ける所」を**通路(road)と階段(stairs)の線だけ**としていて、
そこから 4 m(`walkable_max_m`)より遠い候補を外れ値にし(相互検証)、粒子の重みを下げ(粒子フィルター)、最後に通路の線へ寄せていた(地図への吸着)。
1-3 の学習地点は通路から 6〜7 m なので、指紋の方式が正しく答えても外されていた。

| | どうしたか |
|---|---|
| 歩ける所 | 通路・階段の線に加えて、**通路から部屋・出入口への線**(地図の部屋はすべてこの線 1 本で通路につながっている)、**部屋・出入口の点のまわり**(`room_radius_m`。既定 5 m)、**学習した地点のまわり 2 m**(人が実際に立った所) |
| 地図への吸着 | 部屋・出入口・学習地点の円の中にいる位置は寄せない(部屋の中に出す)。寄せる先にも通路から部屋への線を入れる |
| 表示 | 「歩ける所から遠い」で外した候補に、何 m 離れていたかも出す(各種計算方法の生表示) |
| パラメータ | `room_radius_m`(1〜20 m)。管理アプリで変えて配れる。サーバーの `lib/positioning-params.php` と揃えてある |

学習は 3F に 2 地点・各 1 回しかない。**全部の部屋と、廊下の 3〜5 m ごとに学習登録する**(各地点で向きを変えて 3 回)と大きく良くなる。
近くの AP(1 台の BSSID を全部)をルーターとして正しい場所に登録し、1 つの階に 3 台以上あると、三辺測量も使える。

## 16. AR 実測の跡の直し(2026-10-08)

1F の実測(印 2 つ・追跡切れ 3 回)で、跡が地図と大きくずれた。印と印の間は正しく重なっていた(廊下の上向き 1°・5°)が、

- **追跡中に位置が飛んでいた**(0.1 秒で 6〜14 m。ARCore が位置を付け直した)。それを 1 本の線でつないで描いていた
- **印の外で向きが約 50° 回っていた**。印 2 つの当てはめを跡の全部に使っていた

| | どうしたか |
|---|---|
| 位置の飛び | 1 回(0.1 秒)に 1 m を超え、4 m/秒を超える動きを「位置の飛び」として記録(`jumpAt`)。ふつうに歩くと 99% が 1.4 m/秒以下だった |
| 区間 | 追跡切れと位置の飛びで、軌跡を区間に分ける。**地図への重ね方は区間ごと**(区間に 2 つ以上の地点の印が要る)。重ねられない区間は描かず、結果の画面に「区間 N 個・M m」と出す |
| 案内 | 始める画面に「5 m ごとと曲がり角で印を付ける・切れ目の直後は近くの地点に印を付け直す」。実測中の帯に「前の印から N m」、8 m を超えたら黄色で印を促す |
| 方位のずれ | 1 回の追跡の中で 68° → 263° と半回転していた。端末がほぼ立っているときだけ使い、45° を超える外れは 3 秒続くまで取り込まない。追跡切れ・位置の飛びのあとは求め直す。調べるため、軌跡にその瞬間の値も残す(5 列目) |

## 17. RealityScan で 3D を作る(2026-10-08)

AR 実測の結果を出すと、管理アプリが**重ね方(alignment)**を記録に書き足してサーバーへ送り直す。
区間ごとに「AR の座標 → 東・北(m)」の回転と平行移動で、原点は最初に印を付けた地点、北は地図の補正の「地図の上の方角」。
計算は端末と PC で、サーバーは置いて zip にするだけ。

1. 管理画面「AR の撮影」→「zip を落とす」
2. `scripts\ar-blur-faces.ps1 -Zip <zip>` で展開して顔をぼかす
3. `scripts\ar-realityscan-prep.ps1 -Dir <展開したフォルダ>` で位置を作る(`<フォルダ>\realityscan\`)
   - `trajectory_local.csv` … `name,x,y,z`(x = 東・y = 北・z = 上。m)
   - `ground_control_local.csv` … 印を付けた地点(地上点)
   - `excluded.txt` … 位置を付けなかった画像(印が足りず重ねられない区間など)
   - `-OriginLat <緯度> -OriginLon <経度> [-OriginAlt <標高>]`(原点の地点。地図アプリなどで調べる)を付けると `trajectory_wgs84.csv`・`ground_control_wgs84.csv` も。
     さらに `-WriteExif` で各画像の EXIF に GPS を書く(exiftool が要る: `winget install OliverBetz.ExifTool`)
4. RealityScan(2.2)に読ませる。どちらか:
   - **(A) ローカル**: プロジェクトの座標系(WORKFLOW → Application → Coordinate systems)を `local:1 - Euclidean` にし、
     画像を入れてから Import Metadata の **Trajectory** で `trajectory_local.csv` を読む(書式 Custom `name x y z`・区切りはカンマ・1 行目を読まない・位置の確からしさ 0.3 m 程度)
   - **(B) GPS**: `-WriteExif` で書いた画像を入れる。**マップウィザード**が「GPS のある画像」として数え、そのまま使う
   - どちらでも重なりが悪いときは、マップウィザードの **Ground Control** で `ground_control_*.csv` を読み、**コントロールポイント**を数枚の画像で手で付ける
5. Align → メッシュ → 書き出し(OBJ / glTF)→ Blender で整える
6. 処理が済んだら、管理画面で撮影を消す

**まだ確かめていないこと**(資料で確かめきれなかったので、試して文書を直す):
ローカル座標の Trajectory がマップウィザードで通るか・向きの列なしで足りるか・EXIF の GPS と地上点を併せたときの高さの扱い。
古い撮影(重ね方の無いもの)は、管理アプリの「実測」→「前の記録」から開くと重ね方を付けて送り直すので、そのあと zip を落とし直す。

出典: [Map Wizard](https://rshelp.capturingreality.com/en-US/tools/mapwiz.htm)・[Trajectory Import](https://rshelp.capturingreality.com/en-US/tools/flightlogimport.htm)・[Control Points](https://rshelp.capturingreality.com/en-US/tools/createcontrolpoints.htm)・[Camera Priors](https://rshelp.capturingreality.com/en-US/appbasics/camerasettings_priors.htm)